# Time-Weighted Retrieval (LangChain)

**Step 1 — Install Packages**

We install LangChain and vector DB integrations.

In [ ]:
!pip uninstall -y langchain langchain-community
!pip install -U langchain langchain-community langchain-openai langchain-chroma langchain-experimental

**Step 2 — Imports**

Bring in embeddings, vector DB, documents, and the time-aware retriever.

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-chroma

!pip install \
  langchain==0.1.16 \
  langchain-core==0.1.45 \
  langchain-community==0.1.16 \
  langchain-openai \
  langchain-chroma


In [ ]:
import os
import time
import math

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document


In [ ]:
!pip install -U langchain-openai


In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-chroma chromadb

!pip install \
  langchain==0.2.16 \
  langchain-core==0.2.38 \
  langchain-community==0.2.16 \
  langchain-openai \
  chromadb


In [ ]:
!pip show langchain-community


In [ ]:
import os
import time
import math

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document



**Step 3 — API Config**

Set API key + backend endpoint.

In [ ]:
os.environ["OPENAI_API_KEY"] = "sk-sL7iiJsgPUQAaJXHjDAzGg"
os.environ["OPENAI_API_BASE"] = "https://apidev.navigatelabsai.com"


**Step 4 — Documents With Timestamps**

Each document gets a timestamp so recency can be scored.

In [ ]:
now = time.time()

docs = [
    Document(
        page_content="Scientists clone dinosaurs in a lab.",
        metadata={"timestamp": now - 60 * 60 * 24 * 365},  # 1 year old
    ),
    Document(
        page_content="Breaking news: new dinosaur DNA discovered.",
        metadata={"timestamp": now - 60 * 60 * 2},  # 2 hours old
    ),
    Document(
        page_content="A detective investigates a crime.",
        metadata={"timestamp": now - 60 * 60 * 24 * 30},  # 1 month old
    ),
]


**Step 5 — Embeddings + Vector Store**

Convert text to vectors and store them.

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://apidev.navigatelabsai.com",
)

vectorstore = Chroma.from_documents(docs, embeddings)


**Step 6 — Create Time-Weighted Retriever**

Wrap the vector DB and apply decay to older docs.

In [ ]:
import math
import time

def time_weighted_search(
    query: str,
    k: int = 2,
    fetch_k: int = 6,
    decay_rate: float = 1e-6,
):
    """
    Custom time-weighted retrieval.
    Combines vector similarity + freshness decay.
    """


    results = vectorstore.similarity_search_with_score(query, k=fetch_k)

    now = time.time()
    rescored = []

    for doc, similarity in results:
        age = now - doc.metadata.get("timestamp", now)


        freshness = math.exp(-decay_rate * age)

        final_score = similarity * freshness

        rescored.append((final_score, doc))

    # Step 2: sort by combined score (lower distance = better)
    rescored.sort(key=lambda x: x[0])

    return [doc for _, doc in rescored[:k]]


**Step 7 — Run Query**

Retrieve docs ranked by relevance + freshness.


In [ ]:
query = "dinosaur discovery"

results = time_weighted_search(query)

print("\n--- TIME WEIGHTED RESULTS ---")

for r in results:
    print("CONTENT:", r.page_content)
    print("TIMESTAMP:", r.metadata["timestamp"])
    print("-" * 40)


**Summary**

- Retrieve top-k documents from vector DB

- Check each document’s timestamp

- Apply time-decay to older documents

- Boost newer documents

- Re-rank by relevance + recency

- Return top-k freshest, most relevant docs